# 🐍 Day 3: Mocking

- unittest.mock
- patch
- MagicMock
- side_effect

## Why Mock?

Isolate units under test by replacing dependencies (APIs, DB, files) with controllable fakes.

## MagicMock Basics

MagicMock creates objects that record calls and return configurable values. Any attribute access returns a new MagicMock.

In [ ]:
from unittest.mock import MagicMock

mock = MagicMock()
mock.method.return_value = 42
print(mock.method())
print(mock.method(1, 2, key='val'))
print(mock.method.call_count)
print(mock.method.call_args)

## patch

Temporarily replace an object in a given namespace. Use as context manager or decorator.

In [ ]:
from unittest.mock import patch
import datetime

def get_today():
    return datetime.date.today()

with patch('datetime.date.today') as mock_today:
    mock_today.return_value = datetime.date(2025, 1, 15)
    print(get_today())

print(get_today())  # Real date again

## patch Where It's Used

Patch where the object is looked up, not where it's defined. If `mymodule` imports `requests`, patch `mymodule.requests`.

In [ ]:
# mymodule.py: import requests
# In tests: patch('mymodule.requests'), not 'requests'

def fetch_url(url):
    import urllib.request
    with urllib.request.urlopen(url) as r:
        return r.read().decode()

with patch('urllib.request.urlopen') as mock_urlopen:
    mock_response = MagicMock()
    mock_response.read.return_value = b'Hello'
    mock_response.__enter__.return_value = mock_response
    mock_response.__exit__.return_value = None
    mock_urlopen.return_value = mock_response
    result = fetch_url('http://example.com')
    print(result)

## side_effect

`side_effect` can be: exception to raise, iterable of return values, or callable.

In [ ]:
from unittest.mock import MagicMock

mock = MagicMock()

# Raise exception
mock.bad.side_effect = ValueError("Error")
try:
    mock.bad()
except ValueError as e:
    print(f"Caught: {e}")

# Multiple return values
mock.counter.side_effect = [1, 2, 3]
print(mock.counter(), mock.counter(), mock.counter())

# Callable
mock.add.side_effect = lambda a, b: a + b
print(mock.add(2, 3))

## patch as Decorator

Apply patch to the whole test function.

In [ ]:
from unittest.mock import patch

def get_config():
    import os
    return os.environ.get('CONFIG', 'default')

@patch.dict('os.environ', {'CONFIG': 'test_value'})
def test_config():
    assert get_config() == 'test_value'

test_config()

## patch.object

Patch an attribute on a specific object.

In [ ]:
from unittest.mock import patch, MagicMock

class Service:
    def fetch(self):
        return "real"

with patch.object(Service, 'fetch', return_value='mocked'):
    s = Service()
    print(s.fetch())

## assert_called_with

Verify the mock was called with expected arguments.

In [ ]:
from unittest.mock import MagicMock

mock = MagicMock()
mock.send("hello", to="user")
mock.send.assert_called_once()
mock.send.assert_called_with("hello", to="user")
print("Assertions passed")

## Real Example: Testing API Client

Mock the HTTP layer to test business logic.

In [ ]:
from unittest.mock import patch, MagicMock

def get_user_name(user_id):
    import urllib.request
    import json
    url = f"https://api.example.com/users/{user_id}"
    req = urllib.request.Request(url)
    with urllib.request.urlopen(req) as r:
        data = json.loads(r.read().decode())
    return data.get('name', 'Unknown')

with patch('urllib.request.urlopen') as mock:
    mock_response = MagicMock()
    mock_response.read.return_value = b'{"name": "Alice"}'
    mock_response.__enter__ = lambda s: s
    mock_response.__exit__ = lambda *a: None
    mock.return_value = mock_response
    name = get_user_name(1)
    print(name)

## Common Mistakes

- **Patching wrong location**: Patch where used, not where defined
- **MagicMock vs Mock**: MagicMock supports magic methods (len, iter); Mock doesn't
- **side_effect exhaustion**: Iterable runs out; next call raises StopIteration
- **Not using with/patch as decorator**: Patch may not be restored if test fails

## Exercises

In [ ]:
# Exercise 1: Create a MagicMock that returns 100 when called. Assert it was called once.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use patch to mock time.time() and make it return a fixed value.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Create a mock with side_effect that raises ConnectionError.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use assert_called_with to verify a mock was called with specific args.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Mock open() to return a StringIO so a function that reads a file can be tested.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use patch.dict to temporarily set an environment variable in a test.
# YOUR CODE HERE

## Key Takeaways

- MagicMock: configurable fake, records calls
- patch: replace object in namespace
- side_effect: exception, iterable, or callable
- Patch where the object is used

## 🔜 Next: Day 4 - TDD

Tomorrow: Test-driven development, red-green-refactor!